# Imports 

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, r2_score, classification_report, accuracy_score
import joblib
import warnings

warnings.filterwarnings('ignore')
print("Libraries imported successfully!")

Libraries imported successfully!


# Load the Preprocessed Data

In [4]:
# Load the encoded dataset from Notebook 01
data_path = '../data/processed_freight_data.csv'
df = pd.read_csv(data_path)

print(f"Data loaded successfully! Shape: {df.shape}")

Data loaded successfully! Shape: (2000, 32)


# Define Features (X) and Targets (y)

In [5]:
# CRITICAL: We must drop both targets from our features to prevent data leakage!
X = df.drop(columns=['Delay', 'Risk_Label'])

# Target for Regression (ETA calculation)
y_reg = df['Delay']

# Target for Classification (Risk Level)
y_clf = df['Risk_Label']

# Split into Training and Testing sets (80% train, 20% test)
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

Training features shape: (1600, 30)
Testing features shape: (400, 30)


# Train & Evaluate the Regression Model (Delay/ETA)

In [6]:
print("--- Training ETA (Delay) Regressor ---")

# Initialize and train the Random Forest Regressor
regressor = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
regressor.fit(X_train, y_reg_train)

# Predict on test set
reg_predictions = regressor.predict(X_test)

# Evaluate
mae = mean_absolute_error(y_reg_test, reg_predictions)
r2 = r2_score(y_reg_test, reg_predictions)

print(f"Mean Absolute Error (MAE): {mae:.2f} minutes")
print(f"R-squared Score: {r2:.4f}")
print("(A lower MAE and an R2 closer to 1.0 indicates a better fit.)")

--- Training ETA (Delay) Regressor ---
Mean Absolute Error (MAE): 17.57 minutes
R-squared Score: 0.8214
(A lower MAE and an R2 closer to 1.0 indicates a better fit.)


# Train & Evaluate the Classification Model (Risk Level)

In [7]:
print("--- Training Risk Level Classifier ---")

# Initialize and train the Random Forest Classifier
classifier = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
classifier.fit(X_train, y_clf_train)

# Predict on test set
clf_predictions = classifier.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_clf_test, clf_predictions)
print(f"Accuracy: {accuracy:.4f}\n")
print("Classification Report:")
print(classification_report(y_clf_test, clf_predictions))

--- Training Risk Level Classifier ---
Accuracy: 0.8450

Classification Report:
                precision    recall  f1-score   support

    High Delay       0.84      0.72      0.78       127
Moderate Delay       0.84      0.93      0.88       247
       On-Time       0.94      0.65      0.77        26

      accuracy                           0.84       400
     macro avg       0.88      0.77      0.81       400
  weighted avg       0.85      0.84      0.84       400



# Model Serialization 

In [8]:
import os

# Ensure the models directory exists
os.makedirs('../models', exist_ok=True)

#Save the Regressor
reg_path = '../models/eta_regressor.pkl'
joblib.dump(regressor, reg_path)
print(f"Regressor saved to {reg_path}")

#Save the Classifier
clf_path = '../models/risk_classifier.pkl'
joblib.dump(classifier, clf_path)
print(f"Classifier saved to {clf_path}")

#CRITICAL FOR PRODUCTION: Save the exact feature column names!
# When our FastAPI receives JSON, we must reconstruct a DataFrame with these EXACT columns.
columns_path = '../models/model_columns.pkl'
joblib.dump(list(X.columns), columns_path)
print(f"Model feature columns saved to {columns_path}")

print("\nAll models and dependencies successfully serialized. Ready for deployment!")

Regressor saved to ../models/eta_regressor.pkl
Classifier saved to ../models/risk_classifier.pkl
Model feature columns saved to ../models/model_columns.pkl

All models and dependencies successfully serialized. Ready for deployment!
